# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Rule, in plain words:** a page deserves review first if it is *stale* (no meaningful update in
90+ days) and still *visible* (real search demand) -- with two independent aggravating signals on
top: is it converting search demand into clicks worse than its own position tier's median (a
title/meta problem), and is the traffic it does get failing to engage once someone lands (an
on-page problem)? More-visible pages float higher within any bucket, since a small fix on a
high-traffic page matters more than the same fix on a page nobody sees.

**Score (percentile-ranked, no fitted weights):**

```text
baseline_action_score = 0.45 * visibility_score        # log(impressions_90d), percentile rank
                       + 0.35 * freshness_risk_score    # days_since_last_update, percentile rank
                       + 0.20 * ctr_gap_score            # (position-tier median CTR - page CTR), percentile rank
```

**Reason codes** (coded independently below -- inspired by, not copied from,
`scripts/02_baseline_score.py`):
- `stale_visible_page`: `days_since_last_update >= 90` and `impressions_90d >= 250`
- `thin_visible_page`: `0 < word_count < 1200` and `impressions_90d >= 250`
- `weak_ctr_for_position`: real position data, `avg_position <= 20`, and CTR sits >0.1pp below its
  position tier's median CTR
- `low_engagement_visible`: `sessions_90d >= 30` and `0 < engagement_rate < 30`
- fallback: `general_refresh_review`

Deliberately excluded from both the score AND the reason codes: `trend_direction` / `trend_pct`.
The reference pipeline's baseline uses `trend_direction` in one reason code; I chose not to, so
this rule can be evaluated against `is_declining_label` with none of its inputs being
label-adjacent at all -- a cleaner test of whether stale + visible + underperforming pages really
are the declining ones.


In [1]:
import os, sys
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

import pandas as pd, numpy as np
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
from ml_utils import percentile_rank

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id").copy()
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)  # for evaluation only, never a scoring input

df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])

positioned = df[(df["avg_position"] > 0) & (df["avg_position"] <= 50)]
tier_median_ctr = positioned.groupby("position_tier")["ctr"].median()
df["expected_ctr"] = df["position_tier"].map(tier_median_ctr).fillna(df["ctr"])
df["ctr_gap"] = (df["expected_ctr"] - df["ctr"]).clip(lower=0)
has_position = (df["avg_position"] > 0).astype(int)
df["ctr_gap_score"] = percentile_rank(df["ctr_gap"]) * has_position

df["baseline_action_score"] = (
    0.45 * df["visibility_score"] + 0.35 * df["freshness_risk_score"] + 0.20 * df["ctr_gap_score"]
).clip(0, 1)
print("Score built for", len(df), "pages. Range:", round(df["baseline_action_score"].min(), 3),
      "-", round(df["baseline_action_score"].max(), 3))


Score built for 30000 pages. Range: 0.009 - 0.943


## 2. Build the ranked queue (writes the CSV)

Reason codes, suggested action, rank, and the CSV write.


In [2]:
def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 90 and row["impressions_90d"] >= 250:
        reasons.append("stale_visible_page")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if 0 < row["avg_position"] <= 20 and row["ctr_gap"] > 0.1:
        reasons.append("weak_ctr_for_position")
    if row["sessions_90d"] >= 30 and 0 < row["engagement_rate"] < 30:
        reasons.append("low_engagement_visible")
    if not reasons:
        reasons.append("general_refresh_review")
    return "|".join(reasons)

df["reason_codes"] = df.apply(reason_codes, axis=1)

def suggested_action(reasons):
    r = set(reasons.split("|"))
    if "thin_visible_page" in r:
        return "expand_and_refresh"
    if "weak_ctr_for_position" in r:
        return "refresh_title_and_meta"
    if "stale_visible_page" in r:
        return "refresh"
    return "monitor"

df["suggested_action"] = df["reason_codes"].apply(suggested_action)
df["baseline_rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

out_cols = ["content_id", "client_id", "baseline_rank", "baseline_action_score", "visibility_score",
            "freshness_risk_score", "ctr_gap_score", "reason_codes", "suggested_action",
            "is_declining_label", "impressions_90d", "avg_position", "ctr",
            "days_since_last_update", "word_count"]
queue = df[out_cols].sort_values("baseline_rank")

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
queue.head(5)


Wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_rank,baseline_action_score,visibility_score,freshness_risk_score,ctr_gap_score,reason_codes,suggested_action,is_declining_label,impressions_90d,avg_position,ctr,days_since_last_update,word_count
10870,content_a5dbb404bdc2,client_f369cb89fc,1,0.943460,0.991300,0.9911,0.752450,stale_visible_page|low_engagement_visible,refresh,0,79035,8.7,0.07,106,2691.0
7445,content_c8e9d6ab9013,client_19581e27de,2,0.931243,0.999067,0.8432,0.932717,stale_visible_page|weak_ctr_for_position,refresh_title_and_meta,1,208678,9.7,0.00,104,NaN
9193,content_c1fe78bc4e37,client_19581e27de,3,0.916333,0.997333,0.8432,0.862067,stale_visible_page|weak_ctr_for_position|low_e...,refresh_title_and_meta,1,134055,7.5,0.03,104,NaN
4708,content_b115f7c74779,client_19581e27de,4,0.915853,0.996267,0.8432,0.862067,stale_visible_page|weak_ctr_for_position|low_e...,refresh_title_and_meta,0,123469,8.0,0.03,104,NaN
10646,content_d0cc5baa4995,client_19581e27de,5,0.914068,0.992300,0.8432,0.862067,stale_visible_page|weak_ctr_for_position|low_e...,refresh_title_and_meta,1,83651,6.6,0.03,104,NaN


## 3. Top-20 review

Precision@20 lands at **0.700** against a base rate of 0.542 (computed below) -- 14 of the top 20
are genuinely declining, a real if modest lift over "flip a coin." Reading the actual top 20:
**19 of 20** carry `stale_visible_page` stacked with either `weak_ctr_for_position` (11 pages) or
both `weak_ctr_for_position` + `low_engagement_visible` (8 pages) -- the rule is doing exactly what
it says: surfacing pages that are old, still getting real demand, and visibly underperforming that
demand in one or two concrete ways.

Two representative picks:
- **`content_c8e9d6ab9013`** (rank 2, score 0.931): 208,678 impressions/90d, position 9.7, CTR
  0.00%, 104 days since update, declining. Reason: `stale_visible_page|weak_ctr_for_position`.
  Suggested action: `refresh_title_and_meta`. Confidence: high -- a page-1-adjacent position with
  effectively zero clicks is close to a title/snippet problem by definition, and it IS declining.
  What would make this wrong: if the 0.00% CTR came from a very short measurement window rather
  than a real snippet problem -- worth a manual SERP check before acting.
- **`content_a5dbb404bdc2`** (rank 1, score 0.943): 79,035 impressions/90d, position 8.7, CTR
  0.07%, low engagement -- flagged, but NOT declining (`is_declining_label = 0`). Confidence:
  medium -- it looks exactly like the pages that are declining, so it's a legitimate "not yet, but
  watch it" pick, not an obviously wrong one.


In [3]:
for k in (20, 50):
    p = queue.head(k)["is_declining_label"].mean()
    print(f"Precision@{k}: {p:.3f}   (base rate: {df['is_declining_label'].mean():.3f})")

print()
print(queue[["content_id", "baseline_rank", "baseline_action_score", "reason_codes",
             "suggested_action", "is_declining_label", "impressions_90d", "avg_position", "ctr"]]
      .head(20).to_string(index=False))


Precision@20: 0.700   (base rate: 0.542)
Precision@50: 0.560   (base rate: 0.542)

          content_id  baseline_rank  baseline_action_score                                                    reason_codes       suggested_action  is_declining_label  impressions_90d  avg_position  ctr
content_a5dbb404bdc2              1               0.943460                       stale_visible_page|low_engagement_visible                refresh                   0            79035           8.7 0.07
content_c8e9d6ab9013              2               0.931243                        stale_visible_page|weak_ctr_for_position refresh_title_and_meta                   1           208678           9.7 0.00
content_c1fe78bc4e37              3               0.916333 stale_visible_page|weak_ctr_for_position|low_engagement_visible refresh_title_and_meta                   1           134055           7.5 0.03
content_b115f7c74779              4               0.915853 stale_visible_page|weak_ctr_for_position|low_engag

## 4. Weak picks + leakage check

**Weak picks:** 6 of the top 20 are NOT declining (`is_declining_label = 0`) --
`content_a5dbb404bdc2`, `content_b115f7c74779`, `content_9c8299b55f3c`, `content_36ff89c8214e`,
`content_dd635253d90e`, `content_00a44e45d37c`. All six share the same pattern as the true
positives above: stale, visible, weak CTR for their position -- they are pages that LOOK like
decliners on every observable signal but haven't tipped into a >20% drop yet. That's a reasonable,
explainable miss for a rule using only 3 signals, not a broken rule -- exactly the gap a learned
model gets a chance to close in Week 5, by weighing more signals than 3.

**Leakage check:** the score formula only touches `impressions_90d`, `days_since_last_update`,
`avg_position`, `ctr`, `word_count`, `sessions_90d`, `engagement_rate`, and `position_tier` --
none of which are the label or its sub-windows. Verified below: no column used to build the score
overlaps `trend_direction` / `trend_pct` or the `_last_30d` / `_prev_30d` sub-windows, and no
FlyRank product flag exists in this dataset to leak in the first place.


In [4]:
score_inputs = {"impressions_90d", "days_since_last_update", "avg_position", "ctr", "word_count",
                "sessions_90d", "engagement_rate", "position_tier"}
label_and_subwindows = {"trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d",
                         "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"}
overlap = score_inputs & label_and_subwindows
print("Score inputs / label+subwindow overlap (must be empty):", overlap)

weak = queue.head(20)[queue.head(20)["is_declining_label"] == 0]
print(f"\n{len(weak)} weak picks in the top 20:")
print(weak[["content_id", "baseline_action_score", "reason_codes", "impressions_90d",
            "avg_position", "ctr"]].to_string(index=False))


Score inputs / label+subwindow overlap (must be empty): set()

6 weak picks in the top 20:
          content_id  baseline_action_score                                                    reason_codes  impressions_90d  avg_position  ctr
content_a5dbb404bdc2               0.943460                       stale_visible_page|low_engagement_visible            79035           8.7 0.07
content_b115f7c74779               0.915853 stale_visible_page|weak_ctr_for_position|low_engagement_visible           123469           8.0 0.03
content_9c8299b55f3c               0.910693                        stale_visible_page|weak_ctr_for_position            54783           8.5 0.03
content_36ff89c8214e               0.906677 stale_visible_page|weak_ctr_for_position|low_engagement_visible           295097           7.3 0.05
content_dd635253d90e               0.904128                        stale_visible_page|weak_ctr_for_position            33286           4.6 0.02
content_00a44e45d37c               0.901033  

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.